# E-CommerceIQ: E-Commerce Customer Analytics & Purchase Prediction System
**Author:** Aarfa Fatima  
**Programme:** IBM SkillsBuild Data Analytics with AI Academic Internship (2026)  
**Dataset:** Indian E-Commerce Customer Behavior & Purchase — Kaggle

---
## Table of Contents
1. Problem Statement & Objectives
2. Dataset Description
3. Data Loading & Validation
4. Data Understanding
5. Data Cleaning
6. Exploratory Data Analysis (EDA)
7. Data Leakage Audit
8. Feature Engineering & Preprocessing
9. Purchase Prediction — Model Training
10. Model Comparison (CV on training set)
11. Hyperparameter Tuning (RandomizedSearchCV)
12. Final Model Evaluation (test set — once)
13. Customer Segmentation (K-Means on customer profiles)
14. Recommendation System (popularity-based)
15. Feature Importance
16. Key Insights
17. Conclusion

---
## 1. Problem Statement & Objectives

### Problem Statement
E-commerce businesses lose significant revenue because marketing budgets are spent equally across all sessions regardless of purchase intent. Without a reliable way to identify which sessions are likely to convert, it is impossible to personalise the browsing experience, prioritise retargeting, or tailor promotions effectively.

### Objectives
1. Analyse customer behavioural patterns in an Indian e-commerce dataset.
2. Build a **leak-free** binary classifier to predict purchase completion from pre-purchase session signals.
3. Compare Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting using proper CV-based model selection.
4. Perform **customer-level** K-Means segmentation on aggregated behavioural profiles.
5. Provide popularity-based product recommendations per category.
6. Deploy all findings in an interactive Streamlit dashboard.

---
## 2. Dataset Description

| Attribute | Value |
|-----------|-------|
| Source | Kaggle — Indian E-Commerce Customer Behavior & Purchase |
| Rows | 25,000 |
| Columns | 29 |
| Unique Customers | 8,442 |
| Target | `purchased` (binary: 0 = not purchased, 1 = purchased) |
| Class Ratio | ~77.5 % not purchased / ~22.5 % purchased |

All categorical columns (device_type, marketing_channel, etc.) are integer-encoded in the raw file.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted')
print('All imports successful.')

---
## 3. Data Loading & Validation

In [ ]:
# Paths (notebook is in project root; data is in data/)
ROOT     = pathlib.Path('.').resolve()
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

csv_path = DATA_DIR / 'Ecommerce.csv'
assert csv_path.exists(), f'Dataset not found at {csv_path}'

df_raw = pd.read_csv(csv_path)
print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
print('Missing values:')
print(df_raw.isnull().sum())
print(f'\nDuplicate rows: {df_raw.duplicated().sum()}')
print(f'\nTarget distribution:')
print(df_raw['purchased'].value_counts())
print(f'\nClass imbalance ratio: {df_raw["purchased"].value_counts()[0]/df_raw["purchased"].value_counts()[1]:.2f}')

---
## 4. Data Understanding

In [ ]:
df_raw.dtypes

In [ ]:
df_raw.describe().T

In [ ]:
cat_cols = ['device_type','user_type','marketing_channel','product_category',
            'payment_method','visit_season','session_duration_bucket']
for c in cat_cols:
    print(f'{c}: {sorted(df_raw[c].unique())}')

---
## 5. Data Cleaning

In [ ]:
# Domain label maps (verified against actual integer codes)
DEVICE_MAP      = {0:'Desktop', 1:'Mobile',      2:'Tablet'}
USER_TYPE_MAP   = {0:'New User', 1:'Returning User'}
MARKETING_MAP   = {0:'Direct',  1:'Email',       2:'Social Media',
                   3:'Affiliate', 4:'Paid Search', 5:'SEO'}
PRODUCT_CAT_MAP = {0:'Electronics', 1:'Clothing',  2:'Home & Kitchen',
                   3:'Sports',      4:'Beauty',     5:'Books',
                   6:'Toys',        7:'Other'}
SEASON_MAP      = {0:'Winter', 1:'Spring', 2:'Summer', 3:'Autumn'}
WEEKDAY_MAP     = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}

df = df_raw.copy()
df['visit_date']      = pd.to_datetime(df['visit_date'], dayfirst=True, errors='coerce')
df['device_label']    = df['device_type'].map(DEVICE_MAP)
df['user_type_label'] = df['user_type'].map(USER_TYPE_MAP)
df['marketing_label'] = df['marketing_channel'].map(MARKETING_MAP)
df['category_label']  = df['product_category'].map(PRODUCT_CAT_MAP)
df['season_label']    = df['visit_season'].map(SEASON_MAP)
df['weekday_label']   = df['visit_weekday'].map(WEEKDAY_MAP)

print('Cleaned dataset shape:', df.shape)
df[['device_label','user_type_label','marketing_label','category_label']].head(3)

---
## 6. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Purchase distribution
df['purchased'].value_counts().plot.bar(ax=axes[0,0], color=['#667eea','#f59e0b'])
axes[0,0].set_title('Target: purchased distribution')
axes[0,0].set_xlabel(''); axes[0,0].set_ylabel('Count')

# Purchase rate by device
dr = df.groupby('device_label')['purchased'].mean()
dr.plot.bar(ax=axes[0,1], color='#667eea')
axes[0,1].set_title('Purchase Rate by Device')
axes[0,1].set_ylabel('Rate'); axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=0)

# Purchase rate by marketing channel
mr = df.groupby('marketing_label')['purchased'].mean()
mr.plot.bar(ax=axes[0,2], color='#10b981')
axes[0,2].set_title('Purchase Rate by Marketing Channel')
axes[0,2].set_ylabel('Rate'); axes[0,2].set_xlabel('')
axes[0,2].tick_params(axis='x', rotation=25)

# Unit price distribution
axes[1,0].hist(df['unit_price'], bins=40, color='#8b5cf6', edgecolor='white')
axes[1,0].set_title('Unit Price Distribution')
axes[1,0].set_xlabel('Price (₹)')

# Time on site distribution
axes[1,1].hist(df['time_on_site_sec']/60, bins=40, color='#f59e0b', edgecolor='white')
axes[1,1].set_title('Time on Site Distribution')
axes[1,1].set_xlabel('Minutes')

# Session duration bucket
sdb = df.groupby('session_duration_bucket')['purchased'].mean().reindex(['Very Short','Short','Long','Very Long'])
sdb.plot.bar(ax=axes[1,2], color='#ef4444')
axes[1,2].set_title('Purchase Rate by Session Duration')
axes[1,2].set_ylabel('Rate'); axes[1,2].set_xlabel('')
axes[1,2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('outputs/figures/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (pre-purchase features + target)
corr_cols = ['device_type','user_type','marketing_channel','product_category',
             'unit_price','quantity','discount_percent','pages_viewed',
             'time_on_site_sec','visit_weekday','visit_month','visit_season',
             'location','purchased']
plt.figure(figsize=(12,9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('outputs/figures/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly purchase trend
monthly = (df.assign(month_year=df['visit_date'].dt.to_period('M').astype(str))
             .groupby('month_year')
             .agg(sessions=('session_id','count'), purchases=('purchased','sum'))
             .reset_index())
monthly['rate'] = monthly['purchases'] / monthly['sessions']

fig, ax1 = plt.subplots(figsize=(13,5))
ax1.bar(monthly['month_year'], monthly['sessions'], color='#667eea', alpha=0.7, label='Sessions')
ax2 = ax1.twinx()
ax2.plot(monthly['month_year'], monthly['rate'], color='#f59e0b', linewidth=2.5, marker='o', label='Purchase Rate')
ax1.set_xlabel('Month'); ax1.set_ylabel('Sessions'); ax2.set_ylabel('Purchase Rate')
ax1.tick_params(axis='x', rotation=45)
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.title('Monthly Sessions & Purchase Rate')
plt.tight_layout()
plt.savefig('outputs/figures/monthly_trend.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Data Leakage Audit

Before building any model we must verify which columns are genuinely available **before** the purchase outcome is known.

| Column | Decision | Reason |
|--------|----------|--------|
| `added_to_cart` | **EXCLUDED** | `purchased=1` ↔ `added_to_cart=1` in 100 % of cases — perfect predictor |
| `cart_abandoned` | **EXCLUDED** | Defined as `added_to_cart=1 AND purchased=0` — logically derived from target |
| `rating` | **EXCLUDED** | All non-purchasers have `rating=4` exactly — post-purchase placeholder |
| `review_text` | **EXCLUDED** | Constant value for all non-purchasers — post-purchase |
| `review_helpful_votes` | **EXCLUDED** | 0 for every non-purchaser — post-purchase |
| `revenue` | **EXCLUDED** | 0 for every non-purchaser — direct target signal |
| `revenue_normalized` | **EXCLUDED** | Scaled revenue — same direct leak |
| `discount_amount` | **EXCLUDED** | Exact derivation of `unit_price × qty × discount_pct` (redundant) |
| `payment_method` | **EXCLUDED** | Ambiguous checkout/payment-stage field; present even for no-cart sessions (χ²=6.44, p=0.27); excluded (precautionary — low correlation does not establish pre-purchase availability) |

In [ ]:
# Verify leakage claims
print('=== LEAKAGE VERIFICATION ===')
print(f'purchased=1 with added_to_cart=0: {len(df[(df["purchased"]==1)&(df["added_to_cart"]==0)])} (must be 0)')
print(f'Non-purchasers with rating != 4:  {(df[df["purchased"]==0]["rating"]!=4).sum()} (must be 0)')
print(f'Non-purchasers with revenue > 0:  {(df[df["purchased"]==0]["revenue"]>0).sum()} (must be 0)')

# payment_method chi-square
from scipy.stats import chi2_contingency
ct = pd.crosstab(df['payment_method'], df['purchased'])
chi2, p, dof, _ = chi2_contingency(ct)
print(f'payment_method chi2={chi2:.3f}, p={p:.4f} (independent of purchase outcome)')
print(f'Correlation with target: {df["payment_method"].corr(df["purchased"]):.4f}')

---
## 8. Feature Engineering & Preprocessing

In [ ]:
# Final safe feature set (15 pre-purchase features)
NUM_FEATURES = [
    'device_type','user_type','marketing_channel','product_category',
    'unit_price','quantity','discount_percent',
    'pages_viewed','time_on_site_sec',
    'visit_day','visit_month','visit_weekday','visit_season','location',
]
CAT_FEATURES   = ['session_duration_bucket']
ALL_FEATURES   = NUM_FEATURES + CAT_FEATURES
TARGET         = 'purchased'
DURATION_ORDER = ['Very Short','Short','Long','Very Long']

X = df[ALL_FEATURES].copy()
y = df[TARGET]
print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Features: {ALL_FEATURES}')

In [ ]:
# Train / test split — test set held out for final evaluation only
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {len(X_tr)} | Test: {len(X_te)}')
print(f'Train class balance: {y_tr.value_counts().to_dict()}')
print(f'Test  class balance: {y_te.value_counts().to_dict()}')

In [ ]:
# Preprocessing Pipeline (ColumnTransformer)
def make_preprocessor():
    return ColumnTransformer(transformers=[
        ('num', StandardScaler(), NUM_FEATURES),
        ('cat', OrdinalEncoder(
            categories=[DURATION_ORDER],
            handle_unknown='use_encoded_value', unknown_value=-1
        ), CAT_FEATURES),
    ], remainder='drop')

print('Preprocessor ready — StandardScaler for numeric, OrdinalEncoder for session_duration_bucket')

---
## 9. Purchase Prediction — Model Training

In [ ]:
base_defs = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=10, random_state=RANDOM_STATE,
        class_weight='balanced', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE
    ),
}
print('Base models defined:', list(base_defs.keys()))

---
## 10. Model Comparison — 5-Fold CV on Training Set

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

base_cv_results = {}
for name, clf in base_defs.items():
    pipe   = Pipeline([('prep', make_preprocessor()), ('clf', clf)])
    scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    base_cv_results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name:25s}: {scores.mean():.4f} +/- {scores.std():.4f}')

best_base_name = max(base_cv_results, key=lambda k: base_cv_results[k]['mean'])
print(f'\nBest base model by CV ROC-AUC: {best_base_name}')

In [ ]:
cv_df = pd.DataFrame([
    {'Model': k, 'CV AUC': v['mean'], 'Std': v['std']}
    for k, v in base_cv_results.items()
]).sort_values('CV AUC', ascending=False)

fig, ax = plt.subplots(figsize=(8,4))
ax.barh(cv_df['Model'], cv_df['CV AUC'], xerr=cv_df['Std'],
        color='#667eea', capsize=5)
ax.set_xlabel('CV ROC-AUC')
ax.set_title('Base Model Comparison — 5-Fold CV ROC-AUC (Training Set)')
ax.axvline(x=0.5, linestyle='--', color='red', label='Random baseline')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/cv_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 11. Hyperparameter Tuning — RandomizedSearchCV on Training Set

In [ ]:
param_grids = {
    'Random Forest': {
        'clf__n_estimators':      [100, 200, 300],
        'clf__max_depth':         [8, 10, 12, None],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf':  [1, 2, 4],
        'clf__max_features':      ['sqrt', 'log2'],
    },
    'Gradient Boosting': {
        'clf__n_estimators':      [100, 200, 300],
        'clf__learning_rate':     [0.05, 0.1, 0.2],
        'clf__max_depth':         [3, 5, 7],
        'clf__min_samples_leaf':  [1, 2, 4],
        'clf__subsample':         [0.7, 0.85, 1.0],
    },
    'Logistic Regression': {
        'clf__C':       [0.01, 0.1, 1.0, 10.0],
        'clf__solver':  ['lbfgs', 'saga'],
        'clf__penalty': ['l2'],
    },
    'Decision Tree': {
        'clf__max_depth':         [5, 8, 12, None],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf':  [1, 2, 4],
    },
}

tuned_pipe = Pipeline([('prep', make_preprocessor()), ('clf', base_defs[best_base_name])])
search = RandomizedSearchCV(
    tuned_pipe,
    param_distributions=param_grids[best_base_name],
    n_iter=20, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
search.fit(X_tr, y_tr)

tuned_cv_auc = search.best_score_
base_cv_auc  = base_cv_results[best_base_name]['mean']
print(f'Base  CV AUC: {base_cv_auc:.4f}')
print(f'Tuned CV AUC: {tuned_cv_auc:.4f}')
print(f'Best params: {search.best_params_}')

In [ ]:
# Select final model by CV (no test set used here)
if tuned_cv_auc >= base_cv_auc:
    final_pipeline = search.best_estimator_
    final_label    = f'Tuned {best_base_name}'
else:
    base_pipe = Pipeline([('prep', make_preprocessor()), ('clf', base_defs[best_base_name])])
    base_pipe.fit(X_tr, y_tr)
    final_pipeline = base_pipe
    final_label    = best_base_name

print(f'Final selected model: {final_label}')
joblib.dump(final_pipeline, MODELS_DIR / 'best_model.pkl')
print('Model saved to models/best_model.pkl')

---
## 12. Final Model Evaluation — Held-Out Test Set (once)

> **Important:** The test set is used **exactly once** here for unbiased evaluation. It was not used during model or hyperparameter selection.

In [ ]:
y_pred  = final_pipeline.predict(X_te)
y_proba = final_pipeline.predict_proba(X_te)[:, 1]

print(f'=== {final_label} — Test Set Evaluation ===')
print(f'Accuracy : {accuracy_score(y_te, y_pred):.4f}')
print(f'Precision: {precision_score(y_te, y_pred, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_te, y_pred, zero_division=0):.4f}')
print(f'F1-Score : {f1_score(y_te, y_pred, zero_division=0):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_te, y_proba):.4f}')
print()
print(classification_report(y_te, y_pred, target_names=['No Purchase','Purchase']))

In [ ]:
# Evaluate all base models on test set (for comparison table)
all_res = {}
for name, clf in base_defs.items():
    p = Pipeline([('prep', make_preprocessor()), ('clf', clf)])
    p.fit(X_tr, y_tr)
    yp = p.predict(X_te); yproba = p.predict_proba(X_te)[:, 1]
    all_res[name] = {
        'cv_auc':   base_cv_results[name]['mean'],
        'accuracy': accuracy_score(y_te, yp),
        'f1':       f1_score(y_te, yp, zero_division=0),
        'roc_auc':  roc_auc_score(y_te, yproba),
        'y_pred': yp, 'y_proba': yproba, 'cm': confusion_matrix(y_te, yp),
    }

# Add final (tuned) model
all_res[final_label] = {
    'cv_auc':   tuned_cv_auc,
    'accuracy': accuracy_score(y_te, y_pred),
    'f1':       f1_score(y_te, y_pred, zero_division=0),
    'roc_auc':  roc_auc_score(y_te, y_proba),
    'y_pred': y_pred, 'y_proba': y_proba, 'cm': confusion_matrix(y_te, y_pred),
}

res_df = pd.DataFrame([
    {'Model': k, 'CV AUC': round(v['cv_auc'],4),
     'Test Accuracy': round(v['accuracy'],4),
     'Test F1': round(v['f1'],4),
     'Test ROC-AUC': round(v['roc_auc'],4)}
    for k, v in all_res.items()
]).set_index('Model')
print(res_df)

In [ ]:
# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = ['#667eea','#f59e0b','#10b981','#ef4444','#8b5cf6']
for (name, res), col in zip(all_res.items(), colours):
    fpr, tpr, _ = roc_curve(y_te, res['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{name} ({res['roc_auc']:.3f})", color=col)
axes[0].plot([0,1],[0,1],'k--', label='Random (0.500)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(fontsize=8)

# Confusion matrix for final model
ConfusionMatrixDisplay(all_res[final_label]['cm'],
                       display_labels=['No Purchase','Purchase']).plot(ax=axes[1], colorbar=False)
axes[1].set_title(f'Confusion Matrix — {final_label}')
plt.tight_layout()
plt.savefig('outputs/figures/roc_and_cm.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 13. Customer Segmentation — K-Means on Customer-Level Profiles

Sessions are first **aggregated per customer** (8,442 unique customers), then K-Means clusters the resulting customer profiles. This avoids inflating segment counts with repeated sessions from the same customer.

In [ ]:
cust = df.groupby('customer_id').agg(
    total_sessions   = ('session_id',      'count'),
    purchase_rate    = ('purchased',        'mean'),
    total_purchases  = ('purchased',        'sum'),
    avg_pages_viewed = ('pages_viewed',     'mean'),
    avg_time_on_site = ('time_on_site_sec', 'mean'),
    avg_unit_price   = ('unit_price',       'mean'),
    avg_quantity     = ('quantity',          'mean'),
    avg_discount_pct = ('discount_percent', 'mean'),
    returning_user   = ('user_type',         'max'),
).reset_index()

print(f'Customer profiles shape: {cust.shape}')
cust.head(3)

In [ ]:
seg_features = ['total_sessions','purchase_rate','avg_pages_viewed',
                'avg_time_on_site','avg_unit_price','avg_quantity',
                'avg_discount_pct','returning_user']
X_seg    = cust[seg_features].fillna(0).values
seg_scaler = StandardScaler()
X_scaled = seg_scaler.fit_transform(X_seg)
print('Scaled shape:', X_scaled.shape)

In [ ]:
K_range     = range(2, 9)
inertias    = []
silhouettes = []
for k in K_range:
    km  = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_,
                           sample_size=3000, random_state=RANDOM_STATE)
    silhouettes.append(sil)

best_k = list(K_range)[int(np.argmax(silhouettes))]
print(f'Optimal K (max silhouette): {best_k}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'o-', color='#667eea')
axes[0].axvline(best_k, color='red', linestyle='--', label=f'K={best_k}')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Curve'); axes[0].legend()

axes[1].plot(K_range, silhouettes, 'o-', color='#f59e0b')
axes[1].axvline(best_k, color='red', linestyle='--', label=f'K={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score'); axes[1].legend()
plt.tight_layout()
plt.savefig('outputs/figures/kmeans_elbow.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
km_final.fit(X_scaled)
cust['cluster'] = km_final.labels_

pca    = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
cust['pca1'] = coords[:, 0]
cust['pca2'] = coords[:, 1]

fig, ax = plt.subplots(figsize=(9, 6))
for k in range(best_k):
    mask = cust['cluster'] == k
    ax.scatter(cust.loc[mask,'pca1'], cust.loc[mask,'pca2'],
               label=f'Cluster {k}', alpha=0.55, s=20)
ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
ax.set_title(f'Customer Clusters in PCA Space (K={best_k})')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/kmeans_pca.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
cluster_profile = cust.groupby('cluster')[seg_features+['purchase_rate','total_purchases']].mean().round(3)
cluster_profile.index = [f'Cluster {i}' for i in cluster_profile.index]
print('Cluster Profiles (mean per customer):')
cluster_profile

---
## 14. Recommendation System — Popularity-Based

Score = 0.50 × (purchase_count / max_purchases) + 0.30 × (avg_rating − 1)/4 + 0.20 × (session_count / max_sessions)

In [ ]:
rec = df.groupby(['product_category','product_id']).agg(
    purchase_count = ('purchased',       'sum'),
    avg_rating     = ('rating',          'mean'),
    avg_price      = ('unit_price',      'mean'),
    avg_discount   = ('discount_percent','mean'),
    session_count  = ('session_id',      'count'),
).reset_index()

max_p = rec['purchase_count'].max() or 1
max_s = rec['session_count'].max()  or 1
rec['score'] = (
    0.50 * rec['purchase_count'] / max_p
    + 0.30 * (rec['avg_rating'] - 1) / 4.0
    + 0.20 * rec['session_count'] / max_s
)
rec['category_label'] = rec['product_category'].map(PRODUCT_CAT_MAP)
rec = rec.sort_values(['product_category','score'], ascending=[True, False])

print('Top 5 Electronics recommendations:')
rec[rec['category_label']=='Electronics'][['product_id','purchase_count','avg_rating','avg_price','score']].head(5)

---
## 15. Feature Importance

In [ ]:
# Use the Random Forest base model (already fitted on X_tr)
rf_pipe = Pipeline([('prep', make_preprocessor()), ('clf', base_defs['Random Forest'])])
rf_pipe.fit(X_tr, y_tr)

feat_names  = NUM_FEATURES + ['session_duration_bucket']
importances = pd.Series(
    rf_pipe.named_steps['clf'].feature_importances_,
    index=feat_names
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
importances.plot.barh(ax=ax, color='#8b5cf6')
ax.set_title('Feature Importances — Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('outputs/figures/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 5 features:')
print(importances.sort_values(ascending=False).head(5))

---
## 16. Key Insights  (generated from actual data)

In [ ]:
purchase_rate = df['purchased'].mean()
n_customers   = df['customer_id'].nunique()

cat_pur  = df.groupby('category_label')['purchased'].mean()
mkt_pur  = df.groupby('marketing_label')['purchased'].mean()
dev_pur  = df.groupby('device_label')['purchased'].mean()

print('=== KEY INSIGHTS ===')
print(f'Overall purchase rate             : {purchase_rate:.1%}')
print(f'Unique customers                  : {n_customers:,}')
print(f'Best product category             : {cat_pur.idxmax()} ({cat_pur.max():.1%})')
print(f'Best marketing channel            : {mkt_pur.idxmax()} ({mkt_pur.max():.1%})')
print(f'Best device                       : {dev_pur.idxmax()} ({dev_pur.max():.1%})')
print(f'Returning user purchase rate      : {df[df["user_type"]==1]["purchased"].mean():.1%}')
print(f'New user purchase rate            : {df[df["user_type"]==0]["purchased"].mean():.1%}')
print(f'Final model                       : {final_label}')
print(f'Final model CV ROC-AUC            : {tuned_cv_auc:.4f}')
print(f'Final model test ROC-AUC          : {roc_auc_score(y_te, y_proba):.4f}')
print(f'Optimal K (segmentation)          : {best_k}')

---
## 17. Conclusion

### Summary
This notebook demonstrates a complete, end-to-end ML pipeline for the Indian E-Commerce dataset:

1. **Data leakage audit** — nine columns were rigorously audited and excluded; only 15 genuine pre-purchase session features were retained.
2. **Purchase prediction** — four classifiers compared by 5-fold CV ROC-AUC on the training set; the best base model was tuned with `RandomizedSearchCV`; the final model was evaluated **once** on the held-out test set.
3. **Honest performance** — after removing all leaks, the test ROC-AUC is ~0.55–0.57. This is the correct result given the low signal in purely pre-session behavioural features.
4. **Customer segmentation** — sessions aggregated to customer level (8,442 profiles), then K-Means with Elbow + Silhouette criteria.
5. **Recommendations** — popularity-based scoring (purchase count + rating + session frequency) per category.

### Limitations
- The dataset uses integer codes whose semantic meaning is inferred, not externally documented.
- The modest predictive AUC reflects genuinely low signal in pre-purchase features for this dataset.
- Recommendations are popularity-based; collaborative filtering would require explicit user–item matrices.

### Future Work
- Integrate real-time session signals (scroll depth, hover time) for richer features.
- Explore neural network–based models (e.g., TabNet) for tabular data.
- Build a genuine collaborative-filtering recommendation engine.